# 11 — Interpretability: faithful (RNA side) + domain-localized (protein side)

**What.** Is the cross-attention mechanistically faithful (does it track the model's own ISM attribution), and does the per-residue head's protein attention land on the RRM/KH RNA-binding domains?

**Why.** A model-internal readout on BOTH modalities is the project differentiator.

**Data (Moyon/Marsico lab).** Frozen PARNET `parnet.7m-0.0`, lab binding `binding/{pureclip,narrowpeak_intersect}`, full-223 `encode.filtered.hfds`, ESM/ProtT5/per-residue/STRING, ATtRACT domains. All numbers from committed `mmpartnet_out/*.json`; figures via the reusable `scripts/viz.py` builders, so the same notebook re-plots any teammate's same-schema results. **In-distribution** (all-223 PARNET) unless noted; leave-out PARNET is the decisive follow-up.

## Definitions

RNA-side: attention over positions $a$ vs in-silico mutagenesis $s_i=\tfrac13\sum_{b\neq x_i}|f(x)-f(x_{i\to b})|$; agreement = Spearman $\rho(a,s)$, with a protein-shuffle control. Protein-side: per-residue attention $\alpha$; **domain enrichment** $=\overline{\alpha}_{i\in\mathrm{RRM/KH}}/\overline{\alpha}$ vs a random-window control ($\approx1$).

In [ ]:
import os, sys, json, pathlib
import numpy as np, matplotlib.pyplot as plt
from IPython.display import Markdown, display
_here=pathlib.Path.cwd().resolve()
REPO=next((c for c in (_here,*_here.parents) if (c/'src'/'mmpartnet').is_dir()),_here)
sys.path.insert(0,str(REPO/'scripts')); import viz
OUT=REPO/'mmpartnet_out'; FIGD=REPO/'notebooks'/'demo'/'executed'
def J(n): return json.loads((OUT/n).read_text())
from IPython.display import Image as _Img
def show(fig,name,sup=None):
    if sup: fig.suptitle(sup,fontsize=12.5,fontweight='bold',y=1.02)
    fig.savefig(str(FIGD/name),bbox_inches='tight',dpi=150); plt.close(fig)
    display(_Img(filename=str(FIGD/name)))   # embed the saved PNG inline (backend-independent)
f=J('xattn_faithfulness.json'); p=J('binding_xattn_perres.json'); dr=p['domain_rows']
print(f"RNA-side: Spearman(attn,ISM) real {f['spearman_real'][0]:+.2f} vs shuf {f['spearman_shuf'][0]:+.2f}; real>shuf {f['n_real_gt_shuf']}/{f['n_windows']}")
print(f"Protein-side: RRM/KH enrichment {np.mean([r['enrichment'] for r in dr]):.2f}x vs control {np.mean([r['ctrl_mean'] for r in dr]):.2f}x ({len(dr)} RBPs)")

In [ ]:
import os, sys, json, pathlib
import numpy as np, matplotlib.pyplot as plt
from IPython.display import Markdown, display
_here=pathlib.Path.cwd().resolve()
REPO=next((c for c in (_here,*_here.parents) if (c/'src'/'mmpartnet').is_dir()),_here)
sys.path.insert(0,str(REPO/'scripts')); import viz
OUT=REPO/'mmpartnet_out'; FIGD=REPO/'notebooks'/'demo'/'executed'
def J(n): return json.loads((OUT/n).read_text())
from IPython.display import Image as _Img
def show(fig,name,sup=None):
    if sup: fig.suptitle(sup,fontsize=12.5,fontweight='bold',y=1.02)
    fig.savefig(str(FIGD/name),bbox_inches='tight',dpi=150); plt.close(fig)
    display(_Img(filename=str(FIGD/name)))   # embed the saved PNG inline (backend-independent)
show(viz.fig_faithfulness(J('xattn_faithfulness.json')),'nb11_faithfulness.png','RNA-side: attention is faithful & protein-specific')

In [ ]:
import os, sys, json, pathlib
import numpy as np, matplotlib.pyplot as plt
from IPython.display import Markdown, display
_here=pathlib.Path.cwd().resolve()
REPO=next((c for c in (_here,*_here.parents) if (c/'src'/'mmpartnet').is_dir()),_here)
sys.path.insert(0,str(REPO/'scripts')); import viz
OUT=REPO/'mmpartnet_out'; FIGD=REPO/'notebooks'/'demo'/'executed'
def J(n): return json.loads((OUT/n).read_text())
from IPython.display import Image as _Img
def show(fig,name,sup=None):
    if sup: fig.suptitle(sup,fontsize=12.5,fontweight='bold',y=1.02)
    fig.savefig(str(FIGD/name),bbox_inches='tight',dpi=150); plt.close(fig)
    display(_Img(filename=str(FIGD/name)))   # embed the saved PNG inline (backend-independent)
show(viz.fig_domains(J('binding_xattn_perres.json')),'nb11_domains.png','Protein-side: attention reads the RRM/KH domains')

In [ ]:
import os, sys, json, pathlib
import numpy as np, matplotlib.pyplot as plt
from IPython.display import Markdown, display
_here=pathlib.Path.cwd().resolve()
REPO=next((c for c in (_here,*_here.parents) if (c/'src'/'mmpartnet').is_dir()),_here)
sys.path.insert(0,str(REPO/'scripts')); import viz
OUT=REPO/'mmpartnet_out'; FIGD=REPO/'notebooks'/'demo'/'executed'
def J(n): return json.loads((OUT/n).read_text())
from IPython.display import Image as _Img
def show(fig,name,sup=None):
    if sup: fig.suptitle(sup,fontsize=12.5,fontweight='bold',y=1.02)
    fig.savefig(str(FIGD/name),bbox_inches='tight',dpi=150); plt.close(fig)
    display(_Img(filename=str(FIGD/name)))   # embed the saved PNG inline (backend-independent)
f=J('xattn_faithfulness.json'); dr=J('binding_xattn_perres.json')['domain_rows']
display(Markdown(f'''**Result.** The attention is mechanistically faithful (Spearman {f['spearman_real'][0]:+.2f} real vs {f['spearman_shuf'][0]:+.2f} shuffled, {f['n_real_gt_shuf']}/{f['n_windows']} windows) and the per-residue attention concentrates **{np.mean([r['enrichment'] for r in dr]):.2f}×** in the annotated RRM/KH domains vs ~1.0× control ({sum(r['enrichment']>r['ctrl_mean'] for r in dr)}/{len(dr)} RBPs) — learned unsupervised from binding alone.'''))

## Conclusion

Both interpretability axes hold: position selection tracks the model's own ISM (protein-specifically), and per-residue attention reads the canonical RNA-binding domains. Reported as supplementary per 'attention is not explanation' — beside the ISM ground truth + shuffle control.

Claude-assisted; methods per Horlacher 2023 (RBPNet) + TFBindFormer/CORAL (cross-attention).